# Tokenized Multi-Label Dataset for Legal-BERT

**Purpose**: Preprocess and tokenize the multi-label CUAD dataset for Legal-BERT
fine-tuning. This notebook creates reusable HuggingFace Dataset objects saved to disk.

**This notebook is ONLY preprocessing/tokenization — NO training.**

**Output**: Tokenized train/test datasets ready for `Trainer` API.

---
## Section 1 — Imports & Configuration

In [1]:
import json
import os
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict

# ==========================================
# CONFIGURATION
# ==========================================

RANDOM_SEED = 42
MAX_LENGTH = 256  # reduced from 512 to save memory during training
TEST_SIZE = 0.2
MODEL_NAME = "nlpaueb/legal-bert-base-uncased"

np.random.seed(RANDOM_SEED)

# Paths
BASE_DIR = r"C:\Users\chari\Desktop\Contract_Intelligence_AI"
DATASET_PATH = os.path.join(BASE_DIR, "data", "processed", "multi_label_clause_dataset.csv")
LABEL_MAP_PATH = os.path.join(BASE_DIR, "data", "processed", "label_mapping.json")

TOKENIZED_DIR = os.path.join(BASE_DIR, "data", "processed", "tokenized_multi_label_dataset")
TOKENIZER_SAVE_DIR = os.path.join(BASE_DIR, "models", "legal_bert_multilabel", "tokenizer")

os.makedirs(TOKENIZED_DIR, exist_ok=True)
os.makedirs(TOKENIZER_SAVE_DIR, exist_ok=True)

print("Configuration ready.")
print(f"  Model:       {MODEL_NAME}")
print(f"  Max length:  {MAX_LENGTH}")
print(f"  Test size:   {TEST_SIZE}")
print(f"  Dataset:     {DATASET_PATH}")
print(f"  Tokenized:   {TOKENIZED_DIR}")
print(f"  Tokenizer:   {TOKENIZER_SAVE_DIR}")

Configuration ready.
  Model:       nlpaueb/legal-bert-base-uncased
  Max length:  256
  Test size:   0.2
  Dataset:     C:\Users\chari\Desktop\Contract_Intelligence_AI\data\processed\multi_label_clause_dataset.csv
  Tokenized:   C:\Users\chari\Desktop\Contract_Intelligence_AI\data\processed\tokenized_multi_label_dataset
  Tokenizer:   C:\Users\chari\Desktop\Contract_Intelligence_AI\models\legal_bert_multilabel\tokenizer


---
## Section 2 — Load Dataset

In [2]:
# ==========================================
# LOAD MULTI-LABEL DATASET
# ==========================================

df = pd.read_csv(DATASET_PATH)

with open(LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    label_mapping = json.load(f)

# Identify label columns (everything except text and contract_id)
label_columns = [c for c in df.columns if c not in ["text", "contract_id"]]

print("Dataset Loaded Successfully")
print(f"  Shape:           {df.shape}")
print(f"  Total samples:   {len(df)}")
print(f"  Label columns:   {len(label_columns)}")
print(f"  Unique contracts:{df['contract_id'].nunique()}")
print(f"  Nulls:           {df.isnull().sum().sum()}")

print(f"\nSample rows (first 3):")
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_columns", 8)
display(df.head(3))
pd.reset_option("display.max_colwidth")
pd.reset_option("display.max_columns")

Dataset Loaded Successfully
  Shape:           (20104, 43)
  Total samples:   20104
  Label columns:   41
  Unique contracts:510
  Nulls:           0

Sample rows (first 3):


,text,contract_id,Affiliate License-Licensee,Affiliate License-Licensor,...,Uncapped Liability,Unlimited/All-You-Can-Eat-License,Volume Restriction,Warranty Duration
0,EXHIBIT 10.6 DISTRIBUTOR AGREEMENT THIS DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT,0,0,...,0,0,0,0
1,"mailed, two (2) business days after the date of deposit ...",LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT,0,0,...,0,0,0,0
2,"DISTRIBUTOR AGREEMENT THIS DISTRIBUTOR AGREEMENT (the ""A...",LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT,0,0,...,0,0,0,0


---
## Section 3 — Label Preparation

Each sample needs a **multi-label vector** of length 41.

Example: `[0, 1, 0, 0, 1, 0, ..., 0]`

- `1` = that clause category is present in the snippet
- `0` = that clause category is absent

Labels are stored as **float32** because `BCEWithLogitsLoss` requires float targets.

In [3]:
# ==========================================
# PREPARE LABEL VECTORS
# ==========================================

# Verify label columns match expected count
assert len(label_columns) == 41, f"Expected 41 labels, got {len(label_columns)}"

print(f"Label columns ({len(label_columns)}):")
for i, col in enumerate(label_columns):
    pos = int(df[col].sum())
    print(f"  {i:2d}. {col:<45s} | positives: {pos}")

# Show a sample label vector
sample_labels = df[label_columns].iloc[0].values
print(f"\nSample label vector (row 0):")
print(f"  Length: {len(sample_labels)}")
print(f"  Values: {sample_labels}")
print(f"  Active labels: {int(sample_labels.sum())}")

Label columns (41):
   0. Affiliate License-Licensee                    | positives: 59
   1. Affiliate License-Licensor                    | positives: 23
   2. Agreement Date                                | positives: 468
   3. Anti-Assignment                               | positives: 374
   4. Audit Rights                                  | positives: 214
   5. Cap On Liability                              | positives: 275
   6. Change Of Control                             | positives: 121
   7. Competitive Restriction Exception             | positives: 76
   8. Covenant Not To Sue                           | positives: 100
   9. Document Name                                 | positives: 508
  10. Effective Date                                | positives: 387
  11. Exclusivity                                   | positives: 180
  12. Expiration Date                               | positives: 411
  13. Governing Law                                 | positives: 436
  14. Insurance  

---
## Section 4 — Load Tokenizer

### What is Tokenization?

Transformer models like Legal-BERT cannot process raw text directly. They require
numerical representations. **Tokenization** converts text into a sequence of token IDs:

```
"termination clause" → [2774, 11580] → padded to [2774, 11580, 0, 0, ..., 0]
```

### Key Concepts

- **input_ids**: Integer token IDs from the vocabulary. Each word/subword maps to a
  unique ID. The model uses these as input embeddings.

- **attention_mask**: Binary mask where `1` = real token, `0` = padding token.
  This tells the model which tokens to attend to and which to ignore.

- **Padding**: All sequences must have the same length for batched processing.
  Shorter sequences are padded with `[PAD]` tokens (ID=0) to reach `max_length`.

- **Truncation**: Sequences longer than `max_length` are cut off. We use 256 tokens
  to balance information retention with memory efficiency.

### Why Multi-Label Vectors?

Unlike single-label classification where the target is a single integer (e.g., class 3),
multi-label classification requires a **vector of binary values** — one per label.
Each label is predicted independently using sigmoid activation, not softmax.

In [4]:
# ==========================================
# LOAD LEGAL-BERT TOKENIZER
# ==========================================

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer Loaded Successfully")
print(f"  Model:          {MODEL_NAME}")
print(f"  Vocab size:     {tokenizer.vocab_size}")
print(f"  Max model len:  {tokenizer.model_max_length}")
print(f"  Using max_len:  {MAX_LENGTH}")
print(f"  Pad token:      {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"  CLS token:      {tokenizer.cls_token} (ID: {tokenizer.cls_token_id})")
print(f"  SEP token:      {tokenizer.sep_token} (ID: {tokenizer.sep_token_id})")

# Quick test
test_encoding = tokenizer("This is a termination clause.", truncation=True, padding="max_length", max_length=MAX_LENGTH)
print(f"\nTest encoding:")
print(f"  input_ids length:      {len(test_encoding['input_ids'])}")
print(f"  attention_mask length: {len(test_encoding['attention_mask'])}")
print(f"  First 15 input_ids:    {test_encoding['input_ids'][:15]}")
print(f"  First 15 attn_mask:    {test_encoding['attention_mask'][:15]}")

Tokenizer Loaded Successfully
  Model:          nlpaueb/legal-bert-base-uncased
  Vocab size:     30522
  Max model len:  512
  Using max_len:  256
  Pad token:      [PAD] (ID: 0)
  CLS token:      [CLS] (ID: 101)
  SEP token:      [SEP] (ID: 102)

Test encoding:
  input_ids length:      256
  attention_mask length: 256
  First 15 input_ids:    [101, 226, 223, 145, 474, 849, 117, 102, 0, 0, 0, 0, 0, 0, 0]
  First 15 attn_mask:    [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]


---
## Section 5 — Contract-Level Train/Test Split

> ⚠️ **CRITICAL**: Split by `contract_id`, not by random rows.
> Snippets from the same contract must never appear in both train and test sets.

In [5]:
# ==========================================
# CONTRACT-LEVEL SPLITTING
# ==========================================

def get_contract_level_split(df, test_size=0.2, seed=42):
    """Split dataset at the CONTRACT level to prevent data leakage."""
    rng = np.random.RandomState(seed)
    unique_contracts = df["contract_id"].unique()
    rng.shuffle(unique_contracts)

    split_idx = int(len(unique_contracts) * (1 - test_size))
    train_contracts = set(unique_contracts[:split_idx])
    test_contracts = set(unique_contracts[split_idx:])

    train_df = df[df["contract_id"].isin(train_contracts)].reset_index(drop=True)
    test_df = df[df["contract_id"].isin(test_contracts)].reset_index(drop=True)

    return train_df, test_df, train_contracts, test_contracts


train_df, test_df, train_contracts, test_contracts = get_contract_level_split(
    df, test_size=TEST_SIZE, seed=RANDOM_SEED
)

# Verify zero leakage
overlap = train_contracts & test_contracts

print("CONTRACT-LEVEL SPLIT")
print("=" * 50)
print(f"  Train contracts: {len(train_contracts)}")
print(f"  Test contracts:  {len(test_contracts)}")
print(f"  Overlap:         {len(overlap)}", end="")
if len(overlap) == 0:
    print(" ✓ ZERO LEAKAGE")
else:
    print(" ⚠ LEAKAGE DETECTED!")
print(f"\n  Train samples:   {len(train_df)} ({100*len(train_df)/len(df):.1f}%)")
print(f"  Test samples:    {len(test_df)} ({100*len(test_df)/len(df):.1f}%)")

CONTRACT-LEVEL SPLIT
  Train contracts: 408
  Test contracts:  102
  Overlap:         0 ✓ ZERO LEAKAGE

  Train samples:   16073 (79.9%)
  Test samples:    4031 (20.1%)


---
## Section 6 — Tokenization Pipeline

In [6]:
# ==========================================
# TOKENIZATION FUNCTION
# ==========================================

def tokenize_and_prepare(dataframe, tokenizer, label_columns, max_length=256):
    """Tokenize text and prepare labels for HuggingFace Dataset.

    Args:
        dataframe: DataFrame with 'text' and label columns
        tokenizer: HuggingFace tokenizer
        label_columns: list of label column names
        max_length: max sequence length for padding/truncation

    Returns:
        HuggingFace Dataset with input_ids, attention_mask, labels, contract_id
    """
    # Tokenize all texts
    texts = dataframe["text"].tolist()
    encodings = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors=None,  # return as lists for Dataset creation
    )

    # Extract label vectors as float32 (required by BCEWithLogitsLoss)
    labels = dataframe[label_columns].values.astype(np.float32).tolist()

    # Extract contract_ids for debugging/provenance
    contract_ids = dataframe["contract_id"].tolist()

    # Build dataset dict
    dataset_dict = {
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "labels": labels,
        "contract_id": contract_ids,
    }

    return Dataset.from_dict(dataset_dict)


print("Tokenization function defined.")

Tokenization function defined.


In [7]:
# ==========================================
# TOKENIZE TRAIN SET
# ==========================================

print("Tokenizing train set...")
train_dataset = tokenize_and_prepare(train_df, tokenizer, label_columns, max_length=MAX_LENGTH)
print(f"  ✓ Train dataset: {train_dataset}")
print(f"    Features: {list(train_dataset.features.keys())}")

Tokenizing train set...


  ✓ Train dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'contract_id'],
    num_rows: 16073
})
    Features: ['input_ids', 'attention_mask', 'labels', 'contract_id']


In [8]:
# ==========================================
# TOKENIZE TEST SET
# ==========================================

print("Tokenizing test set...")
test_dataset = tokenize_and_prepare(test_df, tokenizer, label_columns, max_length=MAX_LENGTH)
print(f"  ✓ Test dataset: {test_dataset}")
print(f"    Features: {list(test_dataset.features.keys())}")

Tokenizing test set...


  ✓ Test dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'contract_id'],
    num_rows: 4031
})
    Features: ['input_ids', 'attention_mask', 'labels', 'contract_id']


---
## Section 7 — Dataset Validation

In [9]:
# ==========================================
# COMPREHENSIVE VALIDATION
# ==========================================

print("=" * 60)
print("DATASET VALIDATION")
print("=" * 60)

# --- Shape validation ---
print(f"\n1. SHAPES")
print(f"   Train: {len(train_dataset)} samples")
print(f"   Test:  {len(test_dataset)} samples")

# --- Token length validation ---
print(f"\n2. TOKEN LENGTHS")
sample_train = train_dataset[0]
sample_test = test_dataset[0]
print(f"   input_ids length (train[0]):      {len(sample_train['input_ids'])}")
print(f"   attention_mask length (train[0]):  {len(sample_train['attention_mask'])}")
print(f"   input_ids length (test[0]):       {len(sample_test['input_ids'])}")
assert len(sample_train["input_ids"]) == MAX_LENGTH, "input_ids length mismatch!"
assert len(sample_train["attention_mask"]) == MAX_LENGTH, "attention_mask length mismatch!"
print(f"   ✓ All sequences padded to {MAX_LENGTH}")

# --- Label dimension validation ---
print(f"\n3. LABEL DIMENSIONS")
print(f"   labels length (train[0]): {len(sample_train['labels'])}")
print(f"   labels length (test[0]):  {len(sample_test['labels'])}")
assert len(sample_train["labels"]) == 41, "Label vector size mismatch!"
print(f"   ✓ Label vectors are 41-dimensional")

# --- Label dtype validation ---
print(f"\n4. LABEL DTYPE")
label_val = sample_train["labels"][0]
print(f"   Sample label value: {label_val} (type: {type(label_val).__name__})")
assert isinstance(label_val, float), "Labels must be float for BCEWithLogitsLoss!"
print(f"   ✓ Labels are float type")

# --- Null check ---
print(f"\n5. NULL CHECK")
# Check a batch for None values
has_nulls = False
for i in range(min(100, len(train_dataset))):
    s = train_dataset[i]
    if any(v is None for v in [s["input_ids"], s["attention_mask"], s["labels"]]):
        has_nulls = True
        break
print(f"   Nulls found: {has_nulls}")
print(f"   {'⚠ NULL VALUES DETECTED' if has_nulls else '✓ No nulls in sampled rows'}")

# --- Actual token counts ---
print(f"\n6. ACTUAL TOKEN USAGE (non-padding)")
real_tokens_train = [sum(train_dataset[i]["attention_mask"]) for i in range(min(500, len(train_dataset)))]
print(f"   Sample of {len(real_tokens_train)} train examples:")
print(f"   Min real tokens: {min(real_tokens_train)}")
print(f"   Max real tokens: {max(real_tokens_train)}")
print(f"   Mean real tokens: {np.mean(real_tokens_train):.1f}")
truncated = sum(1 for t in real_tokens_train if t == MAX_LENGTH)
print(f"   Truncated (hit max_length): {truncated} ({100*truncated/len(real_tokens_train):.1f}%)")

print(f"\n{'=' * 60}")
print("VALIDATION COMPLETE ✓")
print(f"{'=' * 60}")

DATASET VALIDATION

1. SHAPES
   Train: 16073 samples
   Test:  4031 samples

2. TOKEN LENGTHS
   input_ids length (train[0]):      256
   attention_mask length (train[0]):  256
   input_ids length (test[0]):       256
   ✓ All sequences padded to 256

3. LABEL DIMENSIONS
   labels length (train[0]): 41
   labels length (test[0]):  41
   ✓ Label vectors are 41-dimensional

4. LABEL DTYPE
   Sample label value: 0.0 (type: float)
   ✓ Labels are float type

5. NULL CHECK
   Nulls found: False
   ✓ No nulls in sampled rows

6. ACTUAL TOKEN USAGE (non-padding)


   Sample of 500 train examples:
   Min real tokens: 48
   Max real tokens: 256
   Mean real tokens: 119.8
   Truncated (hit max_length): 12 (2.4%)

VALIDATION COMPLETE ✓


In [10]:
# ==========================================
# PRINT FULL TOKENIZED SAMPLE
# ==========================================

print("FULLY TOKENIZED SAMPLE (train[0])")
print("=" * 60)
sample = train_dataset[0]
print(f"\ninput_ids (first 30):")
print(f"  {sample['input_ids'][:30]}")
print(f"\nattention_mask (first 30):")
print(f"  {sample['attention_mask'][:30]}")
print(f"\nlabels ({len(sample['labels'])} values):")
print(f"  {sample['labels']}")
print(f"\ncontract_id:")
print(f"  {sample['contract_id']}")

# Decode tokens back to text
decoded = tokenizer.decode(sample["input_ids"], skip_special_tokens=True)
print(f"\nDecoded text (first 200 chars):")
print(f"  {decoded[:200]}...")

# Show active labels
active = [label_columns[i] for i, v in enumerate(sample["labels"]) if v == 1.0]
print(f"\nActive labels: {active if active else '(none — negative sample)'}")

FULLY TOKENIZED SAMPLE (train[0])

input_ids (first 30):
  [101, 675, 258, 117, 203, 2676, 232, 226, 2676, 232, 111, 207, 105, 232, 105, 112, 223, 280, 218, 212, 317, 1864, 1148, 1610, 117, 115, 145, 1278, 469, 111]

attention_mask (first 30):
  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

labels (41 values):
  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

contract_id:
  LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT

Decoded text (first 200 chars):
  exhibit 10. 6 distributor agreement this distributor agreement ( the " agreement " ) is made by and between electric city corp., a delaware corporation ( " company " ) and electric city of illinois ll...

Active labels: ['Document Name']


---
## Section 8 — Save Tokenized Dataset

In [11]:
# ==========================================
# SAVE TOKENIZED DATASETS
# ==========================================

train_save_path = os.path.join(TOKENIZED_DIR, "train")
test_save_path = os.path.join(TOKENIZED_DIR, "test")

train_dataset.save_to_disk(train_save_path)
print(f"✓ Train dataset saved: {train_save_path}")

test_dataset.save_to_disk(test_save_path)
print(f"✓ Test dataset saved: {test_save_path}")

# Verify saved sizes
train_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, fnames in os.walk(train_save_path)
    for f in fnames
)
test_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, fnames in os.walk(test_save_path)
    for f in fnames
)
print(f"\n  Train size on disk: {train_size / (1024*1024):.2f} MB")
print(f"  Test size on disk:  {test_size / (1024*1024):.2f} MB")

Saving the dataset (0/1 shards):   0%|          | 0/16073 [00:00<?, ? examples/s]

✓ Train dataset saved: C:\Users\chari\Desktop\Contract_Intelligence_AI\data\processed\tokenized_multi_label_dataset\train


Saving the dataset (0/1 shards):   0%|          | 0/4031 [00:00<?, ? examples/s]

✓ Test dataset saved: C:\Users\chari\Desktop\Contract_Intelligence_AI\data\processed\tokenized_multi_label_dataset\test

  Train size on disk: 25.97 MB
  Test size on disk:  6.53 MB


In [12]:
# ==========================================
# SAVE TOKENIZER
# ==========================================

tokenizer.save_pretrained(TOKENIZER_SAVE_DIR)
print(f"✓ Tokenizer saved: {TOKENIZER_SAVE_DIR}")

# List saved tokenizer files
for fname in os.listdir(TOKENIZER_SAVE_DIR):
    fpath = os.path.join(TOKENIZER_SAVE_DIR, fname)
    fsize = os.path.getsize(fpath) / 1024
    print(f"  {fname}: {fsize:.1f} KB")

✓ Tokenizer saved: C:\Users\chari\Desktop\Contract_Intelligence_AI\models\legal_bert_multilabel\tokenizer
  tokenizer.json: 685.5 KB
  tokenizer_config.json: 0.4 KB


In [13]:
# ==========================================
# VERIFY RELOAD
# ==========================================

from datasets import load_from_disk

print("Verifying saved datasets can be reloaded...")

reloaded_train = load_from_disk(train_save_path)
reloaded_test = load_from_disk(test_save_path)

print(f"  ✓ Reloaded train: {reloaded_train}")
print(f"  ✓ Reloaded test:  {reloaded_test}")

# Verify content matches
assert len(reloaded_train) == len(train_dataset), "Train size mismatch after reload!"
assert len(reloaded_test) == len(test_dataset), "Test size mismatch after reload!"
assert reloaded_train[0]["input_ids"] == train_dataset[0]["input_ids"], "Content mismatch!"
print(f"  ✓ Content verification passed")

Verifying saved datasets can be reloaded...
  ✓ Reloaded train: Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'contract_id'],
    num_rows: 16073
})
  ✓ Reloaded test:  Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'contract_id'],
    num_rows: 4031
})
  ✓ Content verification passed


---
## Section 9 — Final Summary

In [14]:
print("=" * 60)
print("TOKENIZED MULTI-LABEL DATASET — FINAL SUMMARY")
print("=" * 60)

print(f"\n  Tokenizer:        {MODEL_NAME}")
print(f"  Vocab size:       {tokenizer.vocab_size}")
print(f"  Max seq length:   {MAX_LENGTH}")
print(f"  Number of labels: {len(label_columns)}")

print(f"\n  Train samples:    {len(train_dataset)}")
print(f"  Test samples:     {len(test_dataset)}")
print(f"  Split method:     contract-level (zero leakage)")

print(f"\n  Each sample contains:")
print(f"    - input_ids:      [{MAX_LENGTH}] int tensor")
print(f"    - attention_mask: [{MAX_LENGTH}] int tensor")
print(f"    - labels:         [41] float32 vector")
print(f"    - contract_id:    string (for debugging)")

print(f"\n  Saved locations:")
print(f"    Train: {train_save_path}")
print(f"    Test:  {test_save_path}")
print(f"    Tokenizer: {TOKENIZER_SAVE_DIR}")

print(f"\n  ✓ Tokenized multi-label dataset ready for Legal-BERT training")
print(f"    Next step: multi-label Legal-BERT fine-tuning with:")
print(f"    - sigmoid activation (not softmax)")
print(f"    - BCEWithLogitsLoss")
print(f"    - HuggingFace Trainer API")
print("=" * 60)

TOKENIZED MULTI-LABEL DATASET — FINAL SUMMARY

  Tokenizer:        nlpaueb/legal-bert-base-uncased
  Vocab size:       30522
  Max seq length:   256
  Number of labels: 41

  Train samples:    16073
  Test samples:     4031
  Split method:     contract-level (zero leakage)

  Each sample contains:
    - input_ids:      [256] int tensor
    - attention_mask: [256] int tensor
    - labels:         [41] float32 vector
    - contract_id:    string (for debugging)

  Saved locations:
    Train: C:\Users\chari\Desktop\Contract_Intelligence_AI\data\processed\tokenized_multi_label_dataset\train
    Test:  C:\Users\chari\Desktop\Contract_Intelligence_AI\data\processed\tokenized_multi_label_dataset\test
    Tokenizer: C:\Users\chari\Desktop\Contract_Intelligence_AI\models\legal_bert_multilabel\tokenizer

  ✓ Tokenized multi-label dataset ready for Legal-BERT training
    Next step: multi-label Legal-BERT fine-tuning with:
    - sigmoid activation (not softmax)
    - BCEWithLogitsLoss
    - Huggi